# SharePoint data audit: evidence for moving to Dataverse

This notebook scans the Excel/CSV files in your SharePoint document libraries and writes CSV reports that put numbers on:

| Finding | Why it matters for the Dataverse case |
|---|---|
| **Exact duplicate files** (same bytes, any name) | No single source of truth; storage and confusion |
| **Same data under different names/formats** (e.g. an .xlsx and its .csv export, renamed copies) | Copies drift apart; reports disagree |
| **Overlapping datasets** (one file's rows are mostly inside another) | Manual roll-ups and re-keying |
| **Fragmented tables**: one logical table split across many files (monthly drops), plus **column drift** between them | In Dataverse this is one table; drift is what breaks Power BI refreshes |
| **Shared entities**: the same column (Employee ID, Customer…) repeated across many files | Becomes one Dataverse table referenced by lookups |
| **Repeated attributes inside sheets** (e.g. customer name/region copied on every order row) | Classic normalization redundancy; inconsistent spellings show up as data-quality errors |
| **Hand-built relationships**: VLOOKUP/XLOOKUP/INDEX-MATCH cells, links between workbooks | Dataverse relationships/lookups replace these |
| **Connector limits**: files over 25 MB, sheets over 2,000 rows, sheets near Excel's 1,048,576-row cap | Excel is a weak backend for Power Apps / Power Automate |
| **Wide (pivoted) layouts, mixed-type columns, version-style file names** | Manual reshaping before Power BI; no type enforcement; manual version control |

**Stages** (each one saves its results to disk; see *Save points* below)
1. **Inventory**: walks the folders (metadata only, fast). Skips any folder with "archive" in its name.
2. **Exact duplicates**: hashes file bytes, but only for files that share a size with another file.
3. **Content profiling**: reads one file at a time in row chunks, keeps only small fingerprints and counts, and frees memory after each file.
4. **Analysis**: cross-file comparisons, using only the saved results (you can restart the kernel before this stage).
5. **Outputs**: CSV reports plus a headline summary.

**Save points:** everything is written to `OUTPUT_DIR/_savepoints`. If the kernel dies, memory runs out, or you interrupt a cell, restart the kernel, run the **Configuration** and **Setup** cells, then re-run the stage. It continues where it stopped. `reset_savepoints()` starts over.

**Privacy:** reports contain file paths, sheet names, column headers, counts and hashed fingerprints. No cell values are written out.

## 0. Configuration
The simplest way to give Python access to SharePoint is to **sync the document library with OneDrive** (the *Sync* button in the library toolbar) and point `ROOT_DIRS` at the synced folder. Stage 1 only reads metadata, so cloud-only placeholder files are fine there. Stages 2 and 3 read file contents, which makes OneDrive download those files, so make sure you have enough disk space (or audit one library at a time).

In [ ]:
from pathlib import Path

# One or more synced SharePoint libraries (or any folders) to audit
ROOT_DIRS = [
    Path(r"C:\Users\YOUR_NAME\YOUR_COMPANY\Team Site - Documents"),
]

# Where reports and save points go. Keep this OUTSIDE the synced library.
OUTPUT_DIR = Path.home() / "dataverse_audit"

EXCLUDE_FOLDER_KEYWORDS = ["archive"]   # case-insensitive; a matching folder is skipped with everything under it
INCLUDE_EXTENSIONS = {".csv", ".xlsx"}
COUNT_EXCLUDED_FOLDERS = True           # tally size of skipped folders (metadata only, nothing is opened)

# --- Memory controls ---
CHUNK_CELLS = 1_000_000        # cells held in memory at once while reading a sheet (rows per chunk = this / column count)
FLUSH_EVERY_N_FILES = 20       # write results to the save point at least this often
MEMORY_LIMIT_MB = 1_500        # above this, results are flushed to disk and memory is released straight away
DISTINCT_CAP = 200_000         # exact distinct-value counting per column up to this, then an estimate
ROW_HASH_CAP = 5_000_000       # exact duplicate-row counting per sheet up to this many rows, then an estimate
MAX_ROWS_PER_SHEET = None      # None = read everything; e.g. 300_000 for a quicker first pass
STOP_AFTER_BLANK_ROWS = 5_000  # stop reading a sheet after this many empty rows in a row (formatted-but-empty ranges)
# Optional speed-up: `pip install python-calamine` reads .xlsx 10-20x faster, but it loads one sheet at a
# time into memory (about 11x the file size), so it's only used for files up to this size. Bigger files
# are streamed row by row with openpyxl.
FAST_READER_MAX_MB = 25

# --- Analysis settings ---
SKETCH_K = 256                 # fingerprint size per sheet (larger = more accurate overlap estimates)
COL_SKETCH_K = 64              # fingerprint size per column
FD_SAMPLE_ROWS = 50_000        # rows per sheet used to find repeated attributes (lookup-table candidates)
FD_TOLERANCE = 0.02            # share of keys allowed to have conflicting values and still count (catches typos)
MIN_ROWS_FOR_OVERLAP = 10      # ignore tiny sheets when comparing content
OVERLAP_MIN_JACCARD = 0.5      # report sheet pairs at least this similar...
OVERLAP_MIN_CONTAINMENT = 0.8  # ...or where one sheet's rows are mostly inside the other
SCHEMA_SIMILARITY = 0.7        # header-set similarity (Jaccard) for "same table, drifted columns"...
SCHEMA_SUBSET_SIMILARITY = 0.8 # ...or this share of the smaller header set found in the larger one
MIN_FILES_FOR_SHARED_COLUMN = 3
STALE_AFTER_DAYS = 365

# --- Limits used for flags (Microsoft documentation, checked Nov 2026) ---
POWERAPPS_ROW_LIMIT = 2_000    # Power Apps reads at most 2,000 rows from a non-delegable source such as Excel
EXCEL_CONNECTOR_MAX_MB = 25    # max file size supported by the Excel Online (Business) connector
EXCEL_MAX_ROWS = 1_048_576     # worksheet row limit

## Setup
Imports, save-point helpers and memory helpers. Run this after every kernel restart.

In [ ]:
import os, re, gc, sys, csv, time, math, hashlib, zipfile, itertools, functools, warnings
import datetime as dt
import numpy as np
import pandas as pd
import openpyxl

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except ImportError:
    _PROC = None
try:
    from python_calamine import CalamineWorkbook
except ImportError:
    CalamineWorkbook = None

warnings.filterwarnings("ignore", module="openpyxl")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

SAVE_DIR = OUTPUT_DIR / "_savepoints"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SP = {
    "inventory": SAVE_DIR / "01_inventory.csv",
    "excluded":  SAVE_DIR / "01_excluded_folders.csv",
    "hashes":    SAVE_DIR / "02_hashes.csv",
    "files":     SAVE_DIR / "03_file_profiles.csv",
    "sheets":    SAVE_DIR / "03_sheet_profiles.csv",
    "columns":   SAVE_DIR / "03_column_profiles.csv",
    "lookups":   SAVE_DIR / "03_lookup_candidates.csv",
    "ledger":    SAVE_DIR / "03_done_ledger.csv",
}
STAGE_FILES = {"1": ["inventory", "excluded"], "2": ["hashes"],
               "3": ["files", "sheets", "columns", "lookups", "ledger"]}

# Columns that must always be read back as text (ids, fingerprints, names)
TEXT_COLS = ["file_id", "sha256", "sheet_name", "header", "norm_header", "headers_norm", "headers_sorted",
             "schema_sig", "row_sketch", "col_sketch", "key_column", "dependent_columns", "path",
             "rel_path", "folder", "file_name", "name_base", "error"]


def log(msg):
    print(f"[{dt.datetime.now():%H:%M:%S}] {msg}", flush=True)


def rss_mb():
    """Memory currently used by this notebook process (MB). NaN if psutil isn't installed."""
    return _PROC.memory_info().rss / 1e6 if _PROC is not None else float("nan")


def over_memory_limit():
    return _PROC is not None and rss_mb() > MEMORY_LIMIT_MB


def save_df(df, path):
    """Write atomically, so a crash mid-write can't corrupt a save point."""
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False)
    try:
        os.replace(tmp, path)
    except PermissionError:
        raise PermissionError(f"Can't overwrite {path.name}. Is it open in Excel? Close it and re-run this cell.")


def load_df(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    head = pd.read_csv(path, nrows=0)
    dtypes = {c: str for c in head.columns if c in TEXT_COLS}
    # on_bad_lines="skip": a half-written last line from a crash is ignored
    return pd.read_csv(path, dtype=dtypes, on_bad_lines="skip")


def append_rows(path, rows, columns):
    if not rows:
        return
    df = pd.DataFrame(rows).reindex(columns=columns)
    new_file = not path.exists() or path.stat().st_size == 0
    df.to_csv(path, mode="a", header=new_file, index=False)


def reset_savepoints(stages=("1", "2", "3")):
    """Delete save points for the given stages, e.g. reset_savepoints(["3"]) to re-profile everything."""
    for s in stages:
        for key in STAGE_FILES[str(s)]:
            if SP[key].exists():
                SP[key].unlink()
    log(f"Save points cleared for stage(s) {', '.join(map(str, stages))}")


log(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | openpyxl {openpyxl.__version__} | "
    f"memory {rss_mb():.0f} MB | output: {OUTPUT_DIR}")
if _PROC is None:
    log("psutil isn't installed, so memory-triggered flushing is off (files are still saved every "
        f"{FLUSH_EVERY_N_FILES} files). Run `pip install psutil` to turn it on.")
if CalamineWorkbook is None:
    log("Optional: `pip install python-calamine` makes reading .xlsx files 10-20x faster.")

## Stage 1: Inventory
Walks every root folder, skips "archive" folders, keeps `.csv`/`.xlsx`, and records size, dates and file-name patterns:
- `name_has_version_marker`: v2, final, draft, copy, (1), backup, old… (people doing version control by hand)
- `name_has_date`: 2024-03, Jan, Q1, FY24… (recurring manual exports)
- `name_base`: the name with dates/versions stripped, so `Sales Report - Mar 2024 (FINAL v2).xlsx` and `Copy of Sales Report Feb.xlsx` group together

Re-running loads the save point. Set `REBUILD_INVENTORY = True` to rescan, for example after files change. Changed files get a new `file_id`, so only they are re-hashed and re-profiled.

In [ ]:
REBUILD_INVENTORY = False

_B, _E = r"(?<![a-z0-9])", r"(?![a-z0-9])"
VERSION_RE = re.compile(
    _B + r"(v\d+|ver\s?\d+|version\s?\d+|rev\s?\d+|final|draft|copy|backup|bak|old|updated|latest|revised)" + _E
    + r"|\(\d+\)")
_MONTHS = (r"(january|february|march|april|may|june|july|august|september|october|november|december"
           r"|jan|feb|mar|apr|jun|jul|aug|sep|sept|oct|nov|dec)")
DATE_RE = re.compile("|".join([
    _B + r"(19|20)\d{2}[-_. ]?(0[1-9]|1[0-2])([-_. ]?(0[1-9]|[12]\d|3[01]))?" + _E,
    _B + r"(0?[1-9]|1[0-2])[-_. ](0?[1-9]|[12]\d|3[01])[-_. ]((19|20)\d{2}|\d{2})" + _E,
    _B + _MONTHS + _E,
    _B + r"q[1-4]" + _E,
    _B + r"(w|wk|week)\s?\d{1,2}" + _E,
    _B + r"fy\s?'?\d{2,4}" + _E,
    _B + r"(19|20)\d{2}" + _E,
]))


def name_base(stem):
    s = stem.lower()
    s = re.sub(r"^copy of\s+", "", s)
    s = re.sub(r"\s*-\s*copy(\s*\(\d+\))?$", "", s)
    s = DATE_RE.sub(" ", s)
    s = VERSION_RE.sub(" ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(_B + r"\d+" + _E, " ", s)
    return " ".join(s.split())


def is_excluded_dir(name):
    n = name.lower()
    return any(k.lower() in n for k in EXCLUDE_FOLDER_KEYWORDS)


def is_temp_file(name):
    return name.startswith(("~$", ".~lock", "._"))


def make_file_id(path, size, mtime):
    return "f" + hashlib.sha1(f"{path}|{size}|{int(mtime)}".encode()).hexdigest()[:15]


def build_inventory():
    rows, excluded, errors = [], [], []
    now = dt.datetime.now()
    for ri, root in enumerate(ROOT_DIRS):
        root = Path(root)
        if not root.exists():
            log(f"WARNING: {root} does not exist; skipping")
            continue
        log(f"Scanning {root}")
        for dirpath, dirnames, filenames in os.walk(root, onerror=errors.append):
            keep = []
            for d in dirnames:
                if is_excluded_dir(d):
                    excluded.append(Path(dirpath) / d)
                else:
                    keep.append(d)
            dirnames[:] = keep            # prune: os.walk won't descend into excluded folders
            for fn in filenames:
                ext = os.path.splitext(fn)[1].lower()
                if ext not in INCLUDE_EXTENSIONS or is_temp_file(fn):
                    continue
                p = Path(dirpath) / fn
                try:
                    st = p.stat()
                except OSError as e:
                    errors.append(e)
                    continue
                mtime = dt.datetime.fromtimestamp(st.st_mtime)
                rel_folder = str(Path(dirpath).relative_to(root))
                rows.append(dict(
                    file_id=make_file_id(p, st.st_size, st.st_mtime), root_index=ri, path=str(p),
                    rel_path=str(p.relative_to(root)), folder="(root)" if rel_folder == "." else rel_folder,
                    file_name=fn, ext=ext, size_bytes=st.st_size, size_mb=round(st.st_size / 1e6, 3),
                    modified=mtime.isoformat(timespec="seconds"), days_since_modified=(now - mtime).days,
                    name_has_version_marker=bool(VERSION_RE.search(Path(fn).stem.lower())),
                    name_has_date=bool(DATE_RE.search(Path(fn).stem.lower())),
                    name_base=name_base(Path(fn).stem),
                ))
    inv = pd.DataFrame(rows)
    if inv.empty:
        raise RuntimeError("No .csv/.xlsx files found. Check ROOT_DIRS.")
    inv = inv.drop_duplicates("path").reset_index(drop=True)

    ex_rows = []
    for d in excluded:
        n = b = 0
        if COUNT_EXCLUDED_FOLDERS:
            for dp, _, fns in os.walk(d, onerror=lambda e: None):
                for fn in fns:
                    if os.path.splitext(fn)[1].lower() in INCLUDE_EXTENSIONS and not is_temp_file(fn):
                        try:
                            b += os.stat(os.path.join(dp, fn)).st_size
                            n += 1
                        except OSError:
                            pass
        ex_rows.append(dict(excluded_folder=str(d), matching_files=n, size_mb=round(b / 1e6, 2)))
    if errors:
        log(f"{len(errors)} folders/files couldn't be read (permissions or paths over 260 characters); first: {errors[0]}")
    return inv, pd.DataFrame(ex_rows, columns=["excluded_folder", "matching_files", "size_mb"])


if SP["inventory"].exists() and not REBUILD_INVENTORY:
    inv = load_df(SP["inventory"])
    log(f"Loaded inventory save point: {len(inv):,} files")
else:
    inv, excluded_df = build_inventory()
    save_df(inv, SP["inventory"])
    save_df(excluded_df, SP["excluded"])
    log(f"Inventory: {len(inv):,} files, {inv.size_bytes.sum() / 1e9:.2f} GB | "
        f"skipped {len(excluded_df):,} archive folders "
        f"({excluded_df.matching_files.sum():,} files, {excluded_df.size_mb.sum() / 1e3:.2f} GB)")

display(inv.groupby("ext").agg(files=("file_id", "size"), total_mb=("size_mb", "sum"), largest_mb=("size_mb", "max")))

## Stage 2: Exact duplicates
Hashes the full bytes of every file that shares its exact size with at least one other file. Unique sizes can't be byte-identical, so they're skipped. Reading is streamed in 4 MB blocks, so memory stays flat. Progress is saved every 50 files.

In [ ]:
def sha256_file(path, block=4 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(block), b""):
            h.update(b)
    return h.hexdigest()


def run_stage2():
    inv = load_df(SP["inventory"])
    done = load_df(SP["hashes"])
    done_ids = set(done.file_id) if done is not None else set()
    cand = inv[inv.duplicated("size_bytes", keep=False) & (inv.size_bytes > 0)]
    todo = cand[~cand.file_id.isin(done_ids)]
    log(f"{len(cand):,} files share a size with another file; {len(todo):,} still to hash "
        f"({todo.size_bytes.sum() / 1e9:.2f} GB)")
    buf, t_last = [], time.time()
    try:
        for i, r in enumerate(todo.itertuples(index=False), 1):
            try:
                buf.append(dict(file_id=r.file_id, sha256=sha256_file(r.path), hash_error=""))
            except OSError as e:
                buf.append(dict(file_id=r.file_id, sha256="", hash_error=str(e)[:200]))
            if len(buf) >= 50:
                append_rows(SP["hashes"], buf, ["file_id", "sha256", "hash_error"])
                buf = []
            if time.time() - t_last > 30:
                log(f"  hashed {i:,}/{len(todo):,}")
                t_last = time.time()
    finally:
        append_rows(SP["hashes"], buf, ["file_id", "sha256", "hash_error"])   # save progress even if interrupted
    if not SP["hashes"].exists():
        save_df(pd.DataFrame(columns=["file_id", "sha256", "hash_error"]), SP["hashes"])
    log("Stage 2 complete")


run_stage2()

## Stage 3: Content profiling
For every file, smallest first:
- **.xlsx structure** (read straight from the zip, no data loaded): sheets, Excel Tables, pivot caches, Power Query, links to other workbooks, formula cells, and lookup formulas (VLOOKUP/HLOOKUP/XLOOKUP/LOOKUP/MATCH/XMATCH) and conditional aggregates (SUMIFS/COUNTIFS…).
- **Sheet data**, read in chunks of about `CHUNK_CELLS` cells (`pandas` chunks for .csv; `openpyxl` row-by-row streaming for .xlsx, or `python-calamine` for .xlsx files up to `FAST_READER_MAX_MB` if it's installed). Values are normalized (case, whitespace, number and date formats) so an .xlsx and its .csv export fingerprint the same. Only these are kept per sheet:
  - counts (rows, blank rows, duplicate rows, empty/constant/mixed-type columns, wide date-like headers)
  - a 256-value **row fingerprint** (bottom-k MinHash) for finding the same rows in other files, regardless of column order or header names
  - a 64-value fingerprint per column, for finding shared entities across files
  - repeated-attribute ("lookup table") candidates from up to `FD_SAMPLE_ROWS` rows

**Memory safety:** each chunk is discarded after it's processed, each file's objects are deleted after it finishes, results are appended to disk every `FLUSH_EVERY_N_FILES` files or whenever memory passes `MEMORY_LIMIT_MB`, and a done-ledger records finished files. Interrupting the cell (■) saves progress first. Re-running continues with the next unfinished file.

In [ ]:
HEADER_SCAN_ROWS = 20
MAX_DETERMINANTS = 25
U64 = np.uint64
_EMPTY = np.empty(0, dtype=U64)

# ---------- fingerprint helpers ----------
def bottom_k(a, b, k):
    """Merge two bottom-k sketches (sorted unique uint64 arrays)."""
    return np.union1d(a, b)[:k]


def kmv_estimate(sketch, k):
    """Distinct-count estimate from a bottom-k sketch (exact when the sketch holds the whole set)."""
    if sketch is None or len(sketch) == 0:
        return 0
    if len(sketch) < k:
        return int(len(sketch))
    return int(round((k - 1) / (float(sketch[k - 1]) / 2.0 ** 64)))


def sk_to_hex(a):
    return a.astype(">u8").tobytes().hex() if len(a) else ""


def hex_to_sk(s):
    if not isinstance(s, str) or not s:
        return _EMPTY
    return np.frombuffer(bytes.fromhex(s), dtype=">u8").astype(U64)


# ---------- value normalization (so .xlsx and .csv copies of the same data match) ----------
_NUM_IN_RE = re.compile(r"^[-+]?\$?(?=\.?\d)(\d{1,3}(,\d{3})+|\d*)(\.\d*)?(e[-+]?\d+)?$")
_DATE_IN_RE = re.compile(r"^\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4}([ t]\d{1,2}:\d{2}(:\d{2}(\.\d+)?)?( ?[ap]m)?)?$")
_NUMNORM_RE = r"^-?\d+(\.\d+)?(e[-+]\d+)?$"
_DATENORM_RE = r"^\d{4}-\d{2}-\d{2}"
_TIME_FMTS = ["%H:%M:%S", "%H:%M", "%I:%M:%S %p", "%I:%M %p", "%I:%M:%S%p", "%I:%M%p", "%H:%M:%S.%f"]


def _fmt_dt(d):
    if d.microsecond >= 500_000:
        d += dt.timedelta(seconds=1)
    d = d.replace(microsecond=0)
    return d.date().isoformat() if d.time() == dt.time(0) else d.isoformat(sep=" ")


def _fmt_num(f):
    if f != f or f in (float("inf"), float("-inf")):
        return ""
    return str(int(f)) if f.is_integer() and abs(f) < 1e15 else f"{f:.10g}"


def _parse_date_str(s):
    try:
        return _fmt_dt(dt.datetime.fromisoformat(s))
    except ValueError:
        pass
    date_part, _, time_part = s.partition(" ")
    if "t" in date_part and not time_part:
        date_part, _, time_part = date_part.partition("t")
    fmts = (["%m/%d/%Y", "%m/%d/%y", "%Y/%m/%d"] if "/" in date_part else
            ["%m-%d-%Y", "%m-%d-%y"] if "-" in date_part else ["%d.%m.%Y", "%m.%d.%Y"])
    for f in fmts:
        try:
            d = dt.datetime.strptime(date_part, f)
            break
        except ValueError:
            continue
    else:
        return None
    if time_part:
        for tf in _TIME_FMTS:
            try:
                t = dt.datetime.strptime(time_part, tf).time()
                return _fmt_dt(dt.datetime.combine(d.date(), t))
            except ValueError:
                continue
        return None
    return d.date().isoformat()


@functools.lru_cache(maxsize=300_000)
def _norm_str(v):
    s = v.strip().lower()
    if not s:
        return ""
    if _NUM_IN_RE.match(s):
        try:
            return _fmt_num(float(s.replace(",", "").replace("$", "")))
        except ValueError:
            pass
    if _DATE_IN_RE.match(s):
        d = _parse_date_str(s)
        if d is not None:
            return d
    return " ".join(s.split())


def _norm_value(v):
    if v is None:
        return ""
    t = type(v)
    if t is str:
        return _norm_str(v)
    if t is bool:
        return "true" if v else "false"
    if t is int:
        return str(v)
    if t is float:
        return _fmt_num(v)
    if isinstance(v, dt.datetime):
        return _fmt_dt(v)
    if isinstance(v, dt.date):
        return v.isoformat()
    if isinstance(v, dt.time):
        return v.replace(microsecond=0).isoformat()
    if isinstance(v, (np.floating, float)):
        return _fmt_num(float(v))
    if isinstance(v, (np.integer, int)):
        return str(int(v))
    return _norm_str(str(v))


# ---------- headers ----------
def clean_headers(raw):
    out, named, period, seen = [], [], [], {}
    for j, h in enumerate(raw):
        is_period = isinstance(h, (dt.date, dt.datetime))
        s = "" if h is None else (_fmt_dt(h) if isinstance(h, dt.datetime) else str(h).strip())
        if s.lower().startswith("unnamed:"):     # pandas placeholder for a blank CSV header
            s = ""
        low = s.lower()
        is_period = is_period or (0 < len(low) <= 12 and bool(DATE_RE.search(low))
                                  and len(DATE_RE.sub("", low).strip(" -_/.'")) <= 2)
        is_named = bool(s) and not re.fullmatch(r"(?i)col(umn)?[\s_]?\d+", s)
        if not s:
            s = f"col_{j + 1}"
        k = seen.get(s.lower(), 0)
        seen[s.lower()] = k + 1
        out.append(s if k == 0 else f"{s}_{k + 1}")
        named.append(is_named)
        period.append(is_period)
    return out, named, period


def norm_header(h):
    return re.sub(r"[^a-z0-9]+", "_", str(h).lower()).strip("_")


# ---------- per-sheet accumulator ----------
class SheetProfiler:
    """Accumulates counts and fingerprints chunk by chunk; raw data is never kept."""

    def __init__(self, width):
        self.W = width
        z = lambda: np.zeros(width, dtype=np.int64)
        self.nonnull, self.len_sum, self.num_cnt, self.date_cnt = z(), z(), z(), z()
        self.col_set = [_EMPTY] * width
        self.col_capped = [False] * width
        self.col_sk = [_EMPTY] * width
        self.n_rows = self.blank_rows = 0
        self.row_parts, self.row_n, self.row_exact = [], 0, True
        self.row_sk = _EMPTY
        self.fd_parts, self.fd_n = [], 0

    def add(self, raw_cols):
        n = len(raw_cols[0]) if raw_cols else 0
        if n == 0:
            return
        H = np.zeros(n, dtype=U64)
        any_val = np.zeros(n, dtype=bool)
        hashes = []
        for j, raw in enumerate(raw_cols):
            v = np.array([_norm_value(x) for x in raw], dtype=object)
            mask = v != ""
            h = pd.util.hash_array(v)
            h[~mask] = 0
            H += h                          # order-independent row hash: sum of cell hashes
            any_val |= mask
            hashes.append(h)
            cnt = int(mask.sum())
            if cnt == 0:
                continue
            vs = pd.Series(v[mask], dtype=object).astype(str)
            self.nonnull[j] += cnt
            self.len_sum[j] += int(vs.str.len().sum())
            self.num_cnt[j] += int(vs.str.match(_NUMNORM_RE).sum())
            self.date_cnt[j] += int(vs.str.match(_DATENORM_RE).sum())
            hu = np.unique(h[mask])
            self.col_sk[j] = bottom_k(self.col_sk[j], hu, COL_SKETCH_K)
            if not self.col_capped[j]:
                merged = np.union1d(self.col_set[j], hu)
                if len(merged) > DISTINCT_CAP:
                    self.col_capped[j], self.col_set[j] = True, _EMPTY
                else:
                    self.col_set[j] = merged
            del v, vs, hu
        nb = int(any_val.sum())
        self.blank_rows += n - nb
        if nb == 0:
            return
        self.n_rows += nb
        rh = pd.util.hash_array(H[any_val])   # re-mix the sums
        self.row_sk = bottom_k(self.row_sk, np.unique(rh), SKETCH_K)
        if self.row_exact:
            self.row_parts.append(rh)
            self.row_n += nb
            if self.row_n > ROW_HASH_CAP:
                self.row_exact, self.row_parts = False, []
            elif len(self.row_parts) >= 20:
                self.row_parts = [np.unique(np.concatenate(self.row_parts))]
        if self.fd_n < FD_SAMPLE_ROWS:
            take = min(nb, FD_SAMPLE_ROWS - self.fd_n)
            self.fd_parts.append(np.column_stack([h[any_val][:take] for h in hashes]))
            self.fd_n += take

    def finalize(self):
        distinct, exact = [], []
        for j in range(self.W):
            if self.col_capped[j]:
                distinct.append(kmv_estimate(self.col_sk[j], COL_SKETCH_K)); exact.append(False)
            else:
                distinct.append(len(self.col_set[j])); exact.append(True)
        if self.row_exact:
            d_rows = len(np.unique(np.concatenate(self.row_parts))) if self.row_parts else 0
        else:
            d_rows = kmv_estimate(self.row_sk, SKETCH_K)
        fd = np.vstack(self.fd_parts) if self.fd_parts else np.empty((0, self.W), dtype=U64)
        self.row_parts, self.fd_parts, self.col_set = [], [], []
        return distinct, exact, d_rows, self.row_exact, fd


def type_guess(nonnull, num, date):
    if nonnull == 0:
        return "empty"
    if num / nonnull >= 0.95:
        return "numeric"
    if date / nonnull >= 0.95:
        return "date"
    if (num + date) / nonnull <= 0.05:
        return "text"
    return "mixed"


# ---------- repeated attributes (functional dependencies) ----------
def find_lookup_candidates(fd, named, types, distinct_full, avg_len, n_rows_total):
    """Find key columns whose other attributes repeat on every row (customer -> name, region...).
    Returns greedy, non-overlapping groups so repeated cells aren't double counted."""
    n, W = fd.shape
    if n < 50:
        return []
    codes, card = [], []
    for j in range(W):
        c = pd.factorize(fd[:, j])[0].astype(np.int64)
        c[fd[:, j] == 0] = -1
        codes.append(c)
        card.append(len(np.unique(c[c >= 0])))
    usable = [j for j in range(W) if named[j] and card[j] > 1]
    dets = [j for j in usable if types[j] != "date" and card[j] <= n / 3 and (codes[j] >= 0).mean() >= 0.9]
    dets = sorted(dets, key=lambda j: -card[j])[:MAX_DETERMINANTS]
    found = []
    for a in dets:
        ca = codes[a]
        a_present = int((ca >= 0).sum())
        deps, bad_keys = [], set()
        for b in usable:
            # b can only depend on a if it has (about) no more distinct values; the slack lets typos through
            if b == a or card[b] > card[a] * (1 + FD_TOLERANCE) + 1:
                continue
            cb = codes[b]
            m = (ca >= 0) & (cb >= 0)
            if m.sum() < 0.5 * a_present:
                continue
            base = int(cb.max()) + 1
            u = np.unique(ca[m] * base + cb[m])
            per_a = np.bincount(u // base)
            keys = int((per_a > 0).sum())
            bad = np.nonzero(per_a > 1)[0]
            if keys and len(bad) / keys <= FD_TOLERANCE:
                deps.append(b)
                bad_keys.update(bad.tolist())
        if deps:
            rep_rows = max(int(n_rows_total) - int(distinct_full[a]), 0)
            found.append(dict(a=a, deps=deps, rep_rows=rep_rows, bad=len(bad_keys)))
    found.sort(key=lambda d: -(d["rep_rows"] * len(d["deps"])))
    chosen, covered = [], set()
    for f in found:
        if f["a"] in covered:
            continue
        deps = [b for b in f["deps"] if b not in covered]
        if not deps or f["rep_rows"] == 0:
            continue
        covered.update(deps + [f["a"]])
        chosen.append(dict(a=f["a"], deps=deps, rep_rows=f["rep_rows"], bad=f["bad"],
                           repeated_cells=f["rep_rows"] * len(deps),
                           repeated_bytes=int(sum(avg_len[b] for b in deps) * f["rep_rows"])))
    return chosen


# ---------- .xlsx structure scan (zip contents only) ----------
_F_RE = re.compile(rb"<f(\s[^>]*)?(?:/>|>([^<]*)</f>)")
_REF_RE = re.compile(rb'ref="([A-Z]+)(\d+)(?::([A-Z]+)(\d+))?"')
_T_RE = re.compile(rb't="(?:shared|array)"')
_LOOKUP_RE = re.compile(rb"(?<![A-Za-z0-9_.])(?:_xlfn\.)?(?:VLOOKUP|HLOOKUP|XLOOKUP|LOOKUP|MATCH|XMATCH)\(", re.I)
_CONDAGG_RE = re.compile(rb"(?<![A-Za-z0-9_.])(?:_xlfn\.)?(?:SUMIFS?|COUNTIFS?|AVERAGEIFS?|MAXIFS|MINIFS)\(", re.I)


def _col_num(letters):
    n = 0
    for ch in letters:
        n = n * 26 + (ch - 64)
    return n


def _range_size(m):
    c1, r1, c2, r2 = m.group(1), m.group(2), m.group(3), m.group(4)
    if not c2:
        return 1
    return (abs(int(r2) - int(r1)) + 1) * (abs(_col_num(c2) - _col_num(c1)) + 1)


def _scan_sheet_formulas(z, info, s, block=1 << 20):
    carry = b""
    with z.open(info) as fh:
        while True:
            chunk = fh.read(block)
            buf = carry + chunk
            if chunk:
                cut = buf.rfind(b"<c ")
                if cut <= 0 and len(buf) < (16 << 20):
                    carry = buf
                    continue
                cut = cut if cut > 0 else len(buf)
                work, carry = buf[:cut], buf[cut:]
            else:
                work, carry = buf, b""
            for m in _F_RE.finditer(work):
                s["n_formula_cells"] += 1
                text = m.group(2)
                if not text:
                    continue
                attrs = m.group(1) or b""
                cells = 1
                if _T_RE.search(attrs):
                    r = _REF_RE.search(attrs)
                    cells = _range_size(r) if r else 1
                if _LOOKUP_RE.search(text):
                    s["n_lookup_formula_cells"] += cells
                if _CONDAGG_RE.search(text):
                    s["n_conditional_agg_cells"] += cells
                if b"!" in text:
                    s["n_cross_sheet_formulas"] += 1
            if not chunk:
                break


def scan_xlsx_structure(path):
    s = dict(n_tables=0, n_pivot_caches=0, n_external_links=0, has_data_connections=False,
             has_power_query=False, n_formula_cells=0, n_lookup_formula_cells=0,
             n_conditional_agg_cells=0, n_cross_sheet_formulas=0, xml_uncompressed_mb=0.0, struct_error="")
    try:
        with zipfile.ZipFile(path) as z:
            for info in z.infolist():
                n = info.filename
                s["xml_uncompressed_mb"] += info.file_size / 1e6
                if n.startswith("xl/tables/") and n.endswith(".xml"):
                    s["n_tables"] += 1
                elif n.startswith("xl/pivotCache/pivotCacheDefinition"):
                    s["n_pivot_caches"] += 1
                elif n.startswith("xl/externalLinks/externalLink") and n.endswith(".xml"):
                    s["n_external_links"] += 1
                elif n == "xl/connections.xml":
                    s["has_data_connections"] = True
                elif n.startswith("customXml/item") and n.endswith(".xml") and "Props" not in n:
                    with z.open(info) as fh:
                        head = fh.read(4096)
                    if b"DataMashup" in head or "DataMashup".encode("utf-16-le") in head:
                        s["has_power_query"] = True
                elif n.startswith("xl/worksheets/sheet") and n.endswith(".xml"):
                    _scan_sheet_formulas(z, info, s)
    except (zipfile.BadZipFile, OSError) as e:
        s["struct_error"] = ("not a readable .xlsx (password-protected, corrupt, or a renamed .xls)"
                             if isinstance(e, zipfile.BadZipFile) else f"read error: {e}"[:200])
    s["xml_uncompressed_mb"] = round(s["xml_uncompressed_mb"], 3)
    return s


# ---------- readers: both yield lists of column sequences ----------
def _is_blank(v):
    return v is None or (isinstance(v, str) and not v.strip())


def openpyxl_rows(ws):
    try:
        ws.reset_dimensions()          # don't trust the stored used-range; read what's really there
    except AttributeError:
        pass
    return ws.iter_rows(values_only=True)


def sheet_reader(rows, info):
    """rows: iterator of row tuples/lists. Returns (raw_headers, header_row_number, chunk_generator),
    or None for an empty sheet."""
    it = iter(rows)
    head = [tuple(r) for r in itertools.islice(it, HEADER_SCAN_ROWS)]
    nn = [sum(not _is_blank(v) for v in r) for r in head]
    if not head or max(nn) == 0:
        return None
    thr = max(1, math.ceil(0.5 * max(nn)))
    h_idx = next(i for i, c in enumerate(nn) if c >= thr)   # skips title rows above the header
    trimmed = lambda r: max((k + 1 for k, v in enumerate(r) if not _is_blank(v)), default=0)
    width = max(trimmed(r) for r in head[h_idx:])
    hdr = list(head[h_idx][:width]) + [None] * (width - len(head[h_idx][:width]))
    rows_per_chunk = max(1_000, CHUNK_CELLS // max(width, 1))

    def gen():
        buf, blank_run, taken = [], 0, 0
        pad = (None,) * width
        for r in itertools.chain(head[h_idx + 1:], it):
            if all(_is_blank(v) for v in r):
                blank_run += 1
                info["blank_rows_skipped"] += 1
                if blank_run >= STOP_AFTER_BLANK_ROWS:
                    info["stopped_on_blank_run"] = True
                    break
                continue
            blank_run = 0
            if len(r) > width and not info["cols_beyond_header"]:
                if any(not _is_blank(v) for v in r[width:]):
                    info["cols_beyond_header"] = True
            r = tuple(r)
            buf.append(r[:width] if len(r) >= width else r + pad[len(r):])
            taken += 1
            if MAX_ROWS_PER_SHEET and taken >= MAX_ROWS_PER_SHEET:
                info["truncated"] = True
                break
            if len(buf) >= rows_per_chunk:
                yield list(zip(*buf))
                buf = []
        if buf:
            yield list(zip(*buf))

    return hdr, h_idx + 1, gen()


def _detect_encoding(path):
    with open(path, "rb") as f:
        sample = f.read(1 << 20)
    if sample.startswith((b"\xff\xfe", b"\xfe\xff")):
        return "utf-16", sample
    try:
        sample.decode("utf-8")
        return "utf-8-sig", sample
    except UnicodeDecodeError as e:
        if e.start >= len(sample) - 4:       # just a multi-byte character cut at the sample edge
            return "utf-8-sig", sample
        return "cp1252", sample


def csv_reader(path, info):
    enc, sample = _detect_encoding(path)
    text = sample.decode(enc if enc != "utf-8-sig" else "utf-8", errors="replace")
    try:
        sep = csv.Sniffer().sniff(text[:65536], delimiters=",;\t|").delimiter
    except csv.Error:
        sep = ","
    info.update(csv_encoding=enc, csv_delimiter=repr(sep))
    kw = dict(sep=sep, dtype=str, keep_default_na=False, encoding=enc, encoding_errors="replace",
              on_bad_lines="skip")
    try:
        hdr = list(pd.read_csv(path, nrows=0, **kw).columns)
    except pd.errors.EmptyDataError:
        return None
    if not hdr:
        return None
    rows_per_chunk = max(1_000, CHUNK_CELLS // max(len(hdr), 1))

    def gen():
        taken = 0
        for chunk in pd.read_csv(path, chunksize=rows_per_chunk, **kw):
            if MAX_ROWS_PER_SHEET and taken + len(chunk) > MAX_ROWS_PER_SHEET:
                chunk = chunk.iloc[:MAX_ROWS_PER_SHEET - taken]
                info["truncated"] = True
            taken += len(chunk)
            yield [chunk[c].tolist() for c in chunk.columns]
            del chunk
            if info.get("truncated"):
                break

    return hdr, 1, gen()


# ---------- profile one sheet / one file ----------
SHEET_COLS = ["file_id", "sheet_index", "sheet_name", "sheet_state", "header_row", "n_rows", "n_cols",
              "n_named_cols", "empty_cols", "constant_cols", "mixed_type_cols", "period_header_cols",
              "blank_rows", "distinct_rows", "distinct_rows_exact", "duplicate_rows", "nonnull_cells",
              "text_bytes", "schema_sig", "headers_norm", "headers_sorted", "truncated",
              "stopped_on_blank_run", "cols_beyond_header", "n_lookup_candidates", "repeated_cells",
              "repeated_bytes", "inconsistent_keys", "row_sketch"]
COLUMN_COLS = ["file_id", "sheet_index", "sheet_name", "col_index", "header", "norm_header", "is_named",
               "non_null", "fill_pct", "distinct", "distinct_exact", "avg_len", "type_guess",
               "is_constant", "is_unique", "col_sketch"]
LOOKUP_COLS = ["file_id", "sheet_index", "sheet_name", "key_column", "key_distinct", "dependent_columns",
               "n_dependents", "repeated_rows", "repeated_cells", "repeated_bytes", "inconsistent_keys",
               "sample_rows"]
FILE_COLS = ["file_id", "status", "error", "reader", "n_sheets", "n_data_sheets", "csv_encoding", "csv_delimiter",
             "n_tables", "n_pivot_caches", "n_external_links", "has_data_connections", "has_power_query",
             "n_formula_cells", "n_lookup_formula_cells", "n_conditional_agg_cells", "n_cross_sheet_formulas",
             "xml_uncompressed_mb", "seconds"]


def profile_sheet(file_id, si, sheet_name, sheet_state, reader_out, info):
    raw_hdr, header_row, chunks = reader_out
    headers, named, period = clean_headers(raw_hdr)
    W = len(headers)
    prof = SheetProfiler(W)
    for cols in chunks:
        prof.add(cols)
        del cols
        if over_memory_limit():
            gc.collect()
    distinct, exact, d_rows, d_exact, fd = prof.finalize()
    n = prof.n_rows
    nonnull = prof.nonnull
    avg_len = [float(prof.len_sum[j] / nonnull[j]) if nonnull[j] else 0.0 for j in range(W)]
    types = [type_guess(nonnull[j], prof.num_cnt[j], prof.date_cnt[j]) for j in range(W)]
    norm = [norm_header(h) for h in headers]
    named_nonempty = [norm[j] for j in range(W) if named[j] and nonnull[j] > 0 and norm[j]]
    sorted_set = sorted(set(named_nonempty))
    col_rows = []
    for j in range(W):
        col_rows.append(dict(
            file_id=file_id, sheet_index=si, sheet_name=sheet_name, col_index=j + 1, header=headers[j],
            norm_header=norm[j], is_named=named[j], non_null=int(nonnull[j]),
            fill_pct=round(100 * nonnull[j] / n, 1) if n else 0.0, distinct=int(distinct[j]),
            distinct_exact=exact[j], avg_len=round(avg_len[j], 2), type_guess=types[j],
            is_constant=bool(distinct[j] == 1 and nonnull[j] > 1),
            is_unique=bool(n > 1 and distinct[j] == nonnull[j] == n),
            col_sketch=sk_to_hex(prof.col_sk[j])))
    cands = find_lookup_candidates(fd, named, types, distinct, avg_len, n)
    lk_rows = [dict(file_id=file_id, sheet_index=si, sheet_name=sheet_name, key_column=headers[c["a"]],
                    key_distinct=int(distinct[c["a"]]), dependent_columns=" | ".join(headers[b] for b in c["deps"]),
                    n_dependents=len(c["deps"]), repeated_rows=c["rep_rows"], repeated_cells=c["repeated_cells"],
                    repeated_bytes=c["repeated_bytes"], inconsistent_keys=c["bad"], sample_rows=int(fd.shape[0]))
               for c in cands]
    sheet_row = dict(
        file_id=file_id, sheet_index=si, sheet_name=sheet_name, sheet_state=sheet_state, header_row=header_row,
        n_rows=n, n_cols=W, n_named_cols=len(named_nonempty),
        empty_cols=int((nonnull == 0).sum()),
        constant_cols=sum(r["is_constant"] for r in col_rows),
        mixed_type_cols=sum(t == "mixed" for t in types),
        period_header_cols=int(sum(period)),
        blank_rows=prof.blank_rows + info.get("blank_rows_skipped", 0),
        distinct_rows=int(d_rows), distinct_rows_exact=d_exact, duplicate_rows=max(n - int(d_rows), 0),
        nonnull_cells=int(nonnull.sum()), text_bytes=int(prof.len_sum.sum()),
        schema_sig=("s" + hashlib.sha1("|".join(sorted_set).encode()).hexdigest()[:12]) if sorted_set else "",
        headers_norm="|".join(named_nonempty), headers_sorted="|".join(sorted_set),
        truncated=info.get("truncated", False), stopped_on_blank_run=info.get("stopped_on_blank_run", False),
        cols_beyond_header=info.get("cols_beyond_header", False),
        n_lookup_candidates=len(cands), repeated_cells=sum(c["repeated_cells"] for c in cands),
        repeated_bytes=sum(c["repeated_bytes"] for c in cands),
        inconsistent_keys=sum(c["bad"] for c in cands), row_sketch=sk_to_hex(prof.row_sk))
    del prof, fd
    return sheet_row, col_rows, lk_rows


def _new_info():
    return dict(blank_rows_skipped=0, stopped_on_blank_run=False, cols_beyond_header=False, truncated=False)


def _profile_xlsx_openpyxl(rec, frow, sheets, cols, lks):
    wb = openpyxl.load_workbook(rec.path, read_only=True, data_only=True)
    try:
        frow["n_sheets"] = len(wb.worksheets)
        for si, ws in enumerate(wb.worksheets, 1):
            info = _new_info()
            rdr = sheet_reader(openpyxl_rows(ws), info)
            if rdr is None:
                continue
            s, c, l = profile_sheet(rec.file_id, si, ws.title, getattr(ws, "sheet_state", "visible"), rdr, info)
            sheets.append(s); cols.extend(c); lks.extend(l)
    finally:
        wb.close()
    frow["reader"] = "openpyxl"


def _profile_xlsx_calamine(rec, frow, sheets, cols, lks):
    wb = CalamineWorkbook.from_path(str(rec.path))
    try:
        metas = [m for m in wb.sheets_metadata if "worksheet" in str(m.typ).lower()]
        frow["n_sheets"] = len(metas)
        for si, m in enumerate(metas, 1):
            sh = wb.get_sheet_by_name(m.name)
            rows = sh.iter_rows() if hasattr(sh, "iter_rows") else iter(sh.to_python())
            vis = str(m.visible).lower()
            state = "veryHidden" if "veryhidden" in vis else "hidden" if "hidden" in vis else "visible"
            info = _new_info()
            rdr = sheet_reader(rows, info)
            if rdr is not None:
                s, c, l = profile_sheet(rec.file_id, si, m.name, state, rdr, info)
                sheets.append(s); cols.extend(c); lks.extend(l)
            del sh, rows
    finally:
        wb.close()
    frow["reader"] = "calamine"


def profile_file(rec):
    t0 = time.time()
    frow = dict(file_id=rec.file_id, status="ok", error="", n_sheets=0, n_data_sheets=0)
    sheets, cols, lks = [], [], []
    path = Path(rec.path)
    if rec.ext == ".xlsx":
        frow.update(scan_xlsx_structure(path))
        struct_error = frow.pop("struct_error", "")
        if struct_error:
            frow.update(status="error", error=struct_error)
        elif CalamineWorkbook is not None and rec.size_mb <= FAST_READER_MAX_MB:
            try:
                _profile_xlsx_calamine(rec, frow, sheets, cols, lks)
            except Exception:                       # anything calamine can't handle: stream it instead
                sheets.clear(); cols.clear(); lks.clear()
                _profile_xlsx_openpyxl(rec, frow, sheets, cols, lks)
        else:
            _profile_xlsx_openpyxl(rec, frow, sheets, cols, lks)
    else:
        info = _new_info()
        rdr = csv_reader(path, info)
        frow.update(csv_encoding=info.get("csv_encoding"), csv_delimiter=info.get("csv_delimiter"), n_sheets=1, reader="pandas")
        if rdr is not None:
            s, c, l = profile_sheet(rec.file_id, 1, "(csv)", "visible", rdr, info)
            sheets.append(s); cols.extend(c); lks.extend(l)
    frow["n_data_sheets"] = sum(1 for s in sheets if s["n_rows"] > 0)
    if frow["status"] == "ok" and frow["n_data_sheets"] == 0:
        frow["status"] = "empty"
    frow["seconds"] = round(time.time() - t0, 2)
    return frow, sheets, cols, lks


# ---------- main loop with save points ----------
def run_stage3():
    inv = load_df(SP["inventory"])
    ledger = load_df(SP["ledger"])
    done = set(ledger.file_id) if ledger is not None else set()
    todo = inv[~inv.file_id.isin(done)].sort_values("size_bytes")
    total_mb = todo.size_bytes.sum() / 1e6
    log(f"{len(done):,} files already profiled; {len(todo):,} to go ({total_mb:,.0f} MB)")
    buf = dict(files=[], sheets=[], columns=[], lookups=[], ledger=[])

    def flush(reason):
        append_rows(SP["files"], buf["files"], FILE_COLS)
        append_rows(SP["sheets"], buf["sheets"], SHEET_COLS)
        append_rows(SP["columns"], buf["columns"], COLUMN_COLS)
        append_rows(SP["lookups"], buf["lookups"], LOOKUP_COLS)
        append_rows(SP["ledger"], buf["ledger"], ["file_id", "finished"])   # ledger last = commit
        n = len(buf["ledger"])
        for k in buf:
            buf[k].clear()
        _norm_str.cache_clear()
        mem = (gc.collect(), rss_mb())[1]
        if n:
            log(f"  save point: +{n} files ({reason}); memory now {mem:,.0f} MB")

    t_start = t_log = time.time()
    mb_done, warned = 0.0, False
    try:
        for i, rec in enumerate(todo.itertuples(index=False), 1):
            try:
                frow, s, c, l = profile_file(rec)
            except MemoryError:
                buf["sheets"].clear(); buf["columns"].clear(); buf["lookups"].clear()
                gc.collect()
                frow, s, c, l = (dict(file_id=rec.file_id, status="error",
                                      error="MemoryError: lower CHUNK_CELLS / DISTINCT_CAP or set MAX_ROWS_PER_SHEET"),
                                 [], [], [])
            except Exception as e:
                frow, s, c, l = dict(file_id=rec.file_id, status="error", error=f"{type(e).__name__}: {e}"[:300]), [], [], []
            buf["files"].append(frow); buf["sheets"].extend(s); buf["columns"].extend(c); buf["lookups"].extend(l)
            buf["ledger"].append(dict(file_id=rec.file_id, finished=dt.datetime.now().isoformat(timespec="seconds")))
            del frow, s, c, l
            mb_done += rec.size_bytes / 1e6
            if len(buf["ledger"]) >= FLUSH_EVERY_N_FILES:
                flush("batch")
            elif over_memory_limit():
                flush(f"memory over {MEMORY_LIMIT_MB} MB")
                if over_memory_limit() and not warned:
                    warned = True
                    log("  memory is still high after flushing (Python doesn't always hand memory back). "
                        "Progress is saved: if things slow down, restart the kernel, re-run Configuration + "
                        "Setup, then this cell to continue.")
            if time.time() - t_log > 30:
                rate = mb_done / max(time.time() - t_start, 1e-9)
                eta = (total_mb - mb_done) / rate / 60 if rate else float("nan")
                log(f"  {i:,}/{len(todo):,} files | {mb_done:,.0f}/{total_mb:,.0f} MB | mem {rss_mb():,.0f} MB | ~{eta:,.0f} min left")
                t_log = time.time()
    except KeyboardInterrupt:
        log("Interrupted. Saving progress...")
    finally:
        flush("end of run")
    log("Stage 3 complete" if not todo.empty else "Nothing left to profile")


run_stage3()

## Stage 4: Analysis
This stage reads only the save points, so if memory is tight you can restart the kernel, run **Configuration** and **Setup**, and start here.

In [ ]:
def _bool(s):
    return s.astype(str).str.lower().isin(["true", "1"])


def load_profiles():
    inv = load_df(SP["inventory"])
    hashes = load_df(SP["hashes"])
    files = load_df(SP["files"]); sheets = load_df(SP["sheets"])
    cols = load_df(SP["columns"]); lks = load_df(SP["lookups"])
    ids = set(inv.file_id)
    def clean(df, keys, cols_):
        if df is None:
            return pd.DataFrame(columns=cols_)
        df = df[df.file_id.isin(ids)]
        return df.drop_duplicates(keys, keep="last").reset_index(drop=True)
    files = clean(files, ["file_id"], FILE_COLS)
    sheets = clean(sheets, ["file_id", "sheet_index"], SHEET_COLS)
    cols = clean(cols, ["file_id", "sheet_index", "col_index"], COLUMN_COLS)
    lks = clean(lks, ["file_id", "sheet_index", "key_column"], LOOKUP_COLS)
    hashes = clean(hashes, ["file_id"], ["file_id", "sha256", "hash_error"])
    for c in ["distinct_rows_exact", "truncated", "stopped_on_blank_run", "cols_beyond_header"]:
        sheets[c] = _bool(sheets[c])
    for c in ["is_named", "distinct_exact", "is_constant", "is_unique"]:
        cols[c] = _bool(cols[c])
    for c in ["has_data_connections", "has_power_query"]:
        files[c] = _bool(files[c])
    # sort by path so results don't depend on the order files were processed (e.g. after a resume)
    inv = inv.sort_values("rel_path").reset_index(drop=True)
    rank = dict(zip(inv.file_id, range(len(inv))))
    def by_path(df, extra):
        return (df.assign(_r=df.file_id.map(rank)).sort_values(["_r"] + extra)
                  .drop(columns="_r").reset_index(drop=True))
    files, hashes = by_path(files, []), by_path(hashes, [])
    sheets, lks = by_path(sheets, ["sheet_index"]), by_path(lks, ["sheet_index", "key_column"])
    cols = by_path(cols, ["sheet_index", "col_index"])
    return inv, hashes, files, sheets, cols, lks


inv, hashes, fprof, sheets, cols, lks = load_profiles()
sheets["uid"] = sheets.file_id + ":" + sheets.sheet_index.astype(int).astype(str)
fmeta = inv.set_index("file_id")
log(f"Loaded: {len(inv):,} files, {len(fprof):,} profiled, {len(sheets):,} sheets, {len(cols):,} columns")


def where(uid_or_fid):
    fid = uid_or_fid.split(":")[0]
    return fmeta.at[fid, "rel_path"]


# ---------- 4a. exact duplicate files ----------
d = inv.merge(hashes[["file_id", "sha256"]], on="file_id", how="left")
d = d[d.sha256.notna() & (d.sha256 != "")]
d = d[d.duplicated("sha256", keep=False)]
exact_groups = (d.groupby("sha256")
                .agg(n_copies=("file_id", "size"), size_mb=("size_mb", "first"),
                     distinct_names=("file_name", "nunique"), folders=("folder", "nunique"),
                     newest=("modified", "max"), oldest=("modified", "min"),
                     files=("rel_path", lambda s: " | ".join(s.head(10))))
                .reset_index().sort_values(["n_copies", "size_mb"], ascending=False))
exact_groups["wasted_mb"] = ((exact_groups.n_copies - 1) * exact_groups.size_mb).round(3)
exact_groups.insert(0, "dup_group", [f"D{i:04d}" for i in range(1, len(exact_groups) + 1)])
file_dup_group = d.merge(exact_groups[["sha256", "dup_group", "n_copies"]], on="sha256")[["file_id", "dup_group", "n_copies"]]
log(f"Exact duplicates: {len(exact_groups):,} groups, {exact_groups.n_copies.sum():,} files, "
    f"{exact_groups.wasted_mb.sum():,.1f} MB in extra copies")

# ---------- 4b. same data in different files, and overlapping datasets ----------
CAND_K, MAX_POSTING = 64, 300


def overlap_metrics(a, b, k, n_a, n_b):
    """Estimated Jaccard similarity and containment of A in B / B in A, from bottom-k sketches."""
    if len(a) == 0 or len(b) == 0:
        return 0.0, 0.0, 0.0, 0
    U = np.union1d(a, b)[:k]
    both = np.isin(U, a, assume_unique=True) & np.isin(U, b, assume_unique=True)
    hits = int(both.sum())
    J = hits / len(U)
    inf = np.iinfo(U64).max
    tau = min(a[-1] if len(a) >= k else inf, b[-1] if len(b) >= k else inf)
    sa, sb = a[a <= tau], b[b <= tau]
    inter_est = J / (1 + J) * (n_a + n_b) if J > 0 else 0.0
    c_ab = np.isin(sa, b).mean() if len(sa) >= 5 else min(1.0, inter_est / max(n_a, 1))
    c_ba = np.isin(sb, a).mean() if len(sb) >= 5 else min(1.0, inter_est / max(n_b, 1))
    return float(J), float(c_ab), float(c_ba), hits


S = sheets[(sheets.n_rows >= MIN_ROWS_FOR_OVERLAP) & sheets.row_sketch.notna() & (sheets.row_sketch != "")].copy()
S["content_key"] = [hashlib.md5(f"{s}|{n}".encode()).hexdigest() for s, n in zip(S.row_sketch, S.distinct_rows)]

# identical content (same row fingerprint and same number of distinct rows)
cg = S[S.duplicated("content_key", keep=False)].copy()
cg = cg.merge(inv[["file_id", "rel_path", "ext", "file_name"]], on="file_id")
content_groups = (cg.groupby("content_key")
                  .agg(n_sheets=("uid", "size"), n_files=("file_id", "nunique"), rows=("distinct_rows", "first"),
                       formats=("ext", lambda s: ",".join(sorted(set(s)))), distinct_names=("file_name", "nunique"),
                       locations=("rel_path", lambda s: " | ".join(s.head(10))),
                       sheets_=("sheet_name", lambda s: " | ".join(s.head(10).astype(str))))
                  .reset_index().sort_values(["n_sheets", "rows"], ascending=False))
content_groups.insert(0, "content_group", [f"C{i:04d}" for i in range(1, len(content_groups) + 1)])
content_groups = content_groups.rename(columns={"sheets_": "sheet_names"})
uid_content_group = cg.merge(content_groups[["content_key", "content_group"]], on="content_key")[["uid", "content_group", "content_key"]]

# overlap between different datasets (one representative per identical group)
reps = S.drop_duplicates("content_key").reset_index(drop=True)
sks = [hex_to_sk(x) for x in reps.row_sketch]
pair_rows = []
if len(reps) > 1:
    lens = [min(len(a), CAND_K) for a in sks]
    post = pd.DataFrame({"h": np.concatenate([a[:CAND_K] for a in sks]), "i": np.repeat(np.arange(len(sks)), lens)})
    post = post[post.h.map(post.h.value_counts()) <= MAX_POSTING]
    cand = post.merge(post, on="h")
    cand = cand[cand.i_x < cand.i_y][["i_x", "i_y"]].drop_duplicates()
    log(f"Comparing {len(cand):,} candidate sheet pairs")
    nd = reps.distinct_rows.to_numpy()
    for ia, ib in cand.itertuples(index=False):
        J, cab, cba, hits = overlap_metrics(sks[ia], sks[ib], SKETCH_K, nd[ia], nd[ib])
        if hits < 3 or (J < OVERLAP_MIN_JACCARD and max(cab, cba) < OVERLAP_MIN_CONTAINMENT):
            continue
        T = OVERLAP_MIN_CONTAINMENT
        rel = ("same rows" if J >= 0.98 else "near-identical" if (cab >= T and cba >= T) else
               "A inside B" if cab >= T else "B inside A" if cba >= T else "partial overlap")
        ra, rb = reps.iloc[ia], reps.iloc[ib]
        pair_rows.append(dict(
            sheet_a=ra.uid, file_a=where(ra.uid), sheet_name_a=ra.sheet_name, rows_a=int(ra.distinct_rows),
            sheet_b=rb.uid, file_b=where(rb.uid), sheet_name_b=rb.sheet_name, rows_b=int(rb.distinct_rows),
            jaccard_est=round(J, 3), pct_of_a_in_b=round(100 * cab, 1), pct_of_b_in_a=round(100 * cba, 1),
            relation=rel, same_file=ra.file_id == rb.file_id, same_columns=ra.schema_sig == rb.schema_sig))
del sks
overlaps = pd.DataFrame(pair_rows, columns=["sheet_a", "file_a", "sheet_name_a", "rows_a", "sheet_b", "file_b",
                                            "sheet_name_b", "rows_b", "jaccard_est", "pct_of_a_in_b",
                                            "pct_of_b_in_a", "relation", "same_file", "same_columns"])
overlaps = overlaps.sort_values(["jaccard_est", "rows_a"], ascending=False).reset_index(drop=True)

# which sheets are redundant (their rows already exist elsewhere)?
redundant = {}   # uid -> (fraction contained, where)
members = {}     # representative uid -> every sheet with identical content
for key, grp in uid_content_group.groupby("content_key"):
    uids = sorted(grp.uid)
    for u in uids[1:]:
        redundant[u] = (1.0, uids[0])
for r in reps.itertuples(index=False):
    members[r.uid] = sorted(uid_content_group.loc[uid_content_group.content_key == r.content_key, "uid"]) or [r.uid]
T = OVERLAP_MIN_CONTAINMENT
for r in overlaps.itertuples(index=False):
    a_in, b_in = r.pct_of_a_in_b / 100, r.pct_of_b_in_a / 100
    if a_in >= T and b_in >= T:
        tgt, frac, other = r.sheet_b, b_in, r.sheet_a
    elif a_in >= T:
        tgt, frac, other = r.sheet_a, a_in, r.sheet_b
    elif b_in >= T:
        tgt, frac, other = r.sheet_b, b_in, r.sheet_a
    else:
        continue
    for u in members.get(tgt, [tgt]):          # identical copies of a contained sheet are contained too
        if frac > redundant.get(u, (0.0, None))[0]:
            redundant[u] = (frac, other)
red = pd.DataFrame([(u, f, o) for u, (f, o) in redundant.items()], columns=["uid", "frac_elsewhere", "found_in"])
red = red.merge(sheets[["uid", "file_id", "distinct_rows", "text_bytes"]], on="uid", how="left")
red["redundant_rows_est"] = (red.distinct_rows * red.frac_elsewhere).round()
log(f"Same data in different places: {len(content_groups):,} identical-content groups; "
    f"{len(overlaps):,} overlapping pairs; {len(red):,} sheets whose rows mostly exist elsewhere")

# ---------- 4c. fragmented tables (same columns across many files) and column drift ----------
Tt = sheets[(sheets.n_named_cols >= 3) & (sheets.n_rows > 0) & (sheets.schema_sig.fillna("") != "")].copy()
sig_sets = Tt.drop_duplicates("schema_sig").set_index("schema_sig").headers_sorted.str.split("|").map(frozenset)
parent = {s: s for s in sig_sets.index}


def _find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


if len(sig_sets) > 1:
    ex = sig_sets.explode().reset_index()
    ex.columns = ["sig", "h"]
    ex = ex[ex.h.map(ex.h.value_counts()) <= 500]
    pc = ex.merge(ex, on="h")
    pc = pc[pc.sig_x < pc.sig_y].groupby(["sig_x", "sig_y"]).size().reset_index(name="shared")
    for sx, sy, shared in pc.itertuples(index=False):
        A, B = sig_sets[sx], sig_sets[sy]
        inter = len(A & B)
        jac, sub = inter / len(A | B), inter / min(len(A), len(B))
        if jac >= SCHEMA_SIMILARITY or (sub >= SCHEMA_SUBSET_SIMILARITY and inter >= 3 and jac >= 0.5):
            parent[_find(sx)] = _find(sy)
Tt["schema_root"] = Tt.schema_sig.map(_find)
Tt = Tt.merge(inv[["file_id", "rel_path", "folder", "size_mb", "modified", "name_has_date", "name_base"]], on="file_id")
frag_rows = []
for root, g in Tt.groupby("schema_root"):
    variants = [sig_sets[s] for s in g.schema_sig.unique()]
    common, union = frozenset.intersection(*variants), frozenset.union(*variants)
    files_ = g.drop_duplicates("file_id")
    frag_rows.append(dict(
        schema_root=root, n_files=files_.file_id.nunique(), n_sheets=len(g), n_column_variants=len(variants),
        total_rows=int(g.n_rows.sum()), total_mb=round(files_.size_mb.sum(), 2), n_folders=files_.folder.nunique(),
        pct_files_date_named=round(100 * files_.name_has_date.mean(), 0), n_name_families=files_.name_base.nunique(),
        oldest=files_.modified.min(), newest=files_.modified.max(), n_common_cols=len(common),
        common_columns=" | ".join(sorted(common)[:25]),
        drifting_columns=" | ".join(sorted(union - common)[:25]),
        example_files=" | ".join(files_.rel_path.head(5))))
frag = pd.DataFrame(frag_rows)
if not frag.empty:
    frag = frag.sort_values(["n_files", "total_rows"], ascending=False).reset_index(drop=True)
    frag.insert(0, "logical_table", [f"T{i:04d}" for i in range(1, len(frag) + 1)])
    frag["pattern"] = np.select(
        [(frag.n_files >= 3) & (frag.pct_files_date_named >= 50), frag.n_files >= 2],
        ["recurring drop (date-stamped files)", "same table in several files"], "single file")
    sheet_table = Tt[["uid", "file_id", "schema_root"]].merge(frag[["schema_root", "logical_table", "n_files"]], on="schema_root")
else:
    frag = pd.DataFrame(columns=["logical_table", "schema_root", "n_files", "n_column_variants", "pattern"])
    sheet_table = pd.DataFrame(columns=["uid", "file_id", "schema_root", "logical_table", "n_files"])
multi = frag[frag.n_files >= 2]
log(f"Fragmented tables: {len(multi):,} logical tables are spread over {int(multi.n_files.sum()):,} files; "
    f"{int((frag.n_column_variants > 1).sum()):,} have column drift")

# ---------- 4d. shared entities (same column across files) ----------
GENERIC = {"notes", "note", "comments", "comment", "total", "amount", "value", "qty", "quantity", "count",
           "date", "year", "month", "day", "description", "status", "type", "yes", "no"}
ID_LIKE_RE = re.compile(r"(^|_)(id|code|no|num|number|key|ref|sku|account|acct|emp|employee|customer|cust|vendor)($|_)")
C = cols[cols.is_named & (cols.non_null > 0) & (cols.norm_header.fillna("") != "")]
nf = C.groupby("norm_header").file_id.nunique()
shared_rows = []
for h in nf[nf >= MIN_FILES_FOR_SHARED_COLUMN].index:
    g = C[C.norm_header == h]
    sk_list = [hex_to_sk(x) for x in g.col_sketch]
    merged = _EMPTY
    for a in sk_list:
        merged = bottom_k(merged, a, COL_SKETCH_K)
    union_distinct = max(kmv_estimate(merged, COL_SKETCH_K), 1)
    per_file = g.drop_duplicates("file_id").head(8)
    pj = []
    sk_pf = [hex_to_sk(x) for x in per_file.col_sketch]
    for x, y in itertools.combinations(range(len(sk_pf)), 2):
        pj.append(overlap_metrics(sk_pf[x], sk_pf[y], COL_SKETCH_K, 1, 1)[0])
    total = int(g.non_null.sum())
    main_type = g.type_guess.mode().iat[0]
    shared_rows.append(dict(
        column=h, example_header=g.header.mode().iat[0], n_files=int(g.file_id.nunique()), n_sheets=len(g),
        values_stored=total, distinct_values_est=union_distinct, repetition_factor=round(total / union_distinct, 1),
        avg_pairwise_value_overlap=round(float(np.mean(pj)), 2) if pj else np.nan, main_type=main_type,
        # text columns, or numeric columns named like identifiers (numeric measures such as Hours are excluded)
        entity_candidate=bool((main_type in ("text", "mixed") or (main_type == "numeric" and ID_LIKE_RE.search(h)))
                              and h not in GENERIC and union_distinct >= 2 and total / union_distinct >= 5)))
shared = pd.DataFrame(shared_rows)
if not shared.empty:
    shared = shared.sort_values(["entity_candidate", "n_files", "values_stored"], ascending=False).reset_index(drop=True)
log(f"Shared columns: {len(shared):,} column names appear in ≥{MIN_FILES_FOR_SHARED_COLUMN} files; "
    f"{int(shared.entity_candidate.sum()) if not shared.empty else 0:,} look like shared entities")

## Stage 5: Outputs
Writes the reports to `OUTPUT_DIR`:

| File | One row per | Use it for |
|---|---|---|
| `00_audit_summary.csv` | headline metric | the numbers for your document, each mapped to the Dataverse capability it supports |
| `01_file_audit.csv` | file | **main audit**: every flag for every file; filter it in Excel |
| `02_exact_duplicate_files.csv` | duplicate group | byte-identical copies |
| `03_same_data_different_files.csv` | content group | identical data under different names or formats |
| `04_overlapping_datasets.csv` | sheet pair | one dataset largely inside another |
| `05_fragmented_tables.csv` | logical table | one table spread over many files, and column drift |
| `06_shared_entities.csv` | column name | columns repeated across files (future lookup tables) |
| `07_lookup_table_candidates.csv` | key column in a sheet | repeated attributes a relational model stores once |
| `08_sheet_details.csv`, `09_column_details.csv` | sheet / column | drill-down |
| `10_manual_effort_survey.csv` | process | template for your team's hours, pre-filled with the recurring drops found |

In [ ]:
OUT = OUTPUT_DIR
now = dt.datetime.now()

# ---------- per-file rollup ----------
sh = sheets.merge(sheet_table[["uid", "logical_table"]], on="uid", how="left")
sh_agg = sh.groupby("file_id").agg(
    total_rows=("n_rows", "sum"), max_sheet_rows=("n_rows", "max"), total_cols=("n_cols", "sum"),
    blank_rows=("blank_rows", "sum"), duplicate_rows=("duplicate_rows", "sum"),
    mixed_type_cols=("mixed_type_cols", "sum"), empty_cols=("empty_cols", "sum"),
    wide_layout_sheets=("period_header_cols", lambda s: int((s >= 3).sum())),
    lookup_table_candidates=("n_lookup_candidates", "sum"), repeated_cells=("repeated_cells", "sum"),
    inconsistent_keys=("inconsistent_keys", "sum"), truncated=("truncated", "any"),
    logical_tables=("logical_table", lambda s: " | ".join(sorted(set(s.dropna())))))
red_f = red.groupby("file_id").agg(sheets_with_rows_elsewhere=("uid", "size"),
                                   redundant_rows_est=("redundant_rows_est", "sum"),
                                   rows_found_in=("found_in", lambda s: " | ".join(sorted({where(x) for x in s})[:3])))
cg_f = uid_content_group.assign(file_id=uid_content_group.uid.str.split(":").str[0]).groupby("file_id") \
    .content_group.agg(lambda s: " | ".join(sorted(set(s))))
fa = (inv.drop(columns=["path", "root_index"])
      .merge(file_dup_group, on="file_id", how="left")
      .merge(fprof, on="file_id", how="left")
      .merge(sh_agg, on="file_id", how="left")
      .merge(red_f, on="file_id", how="left")
      .merge(cg_f.rename("content_group"), on="file_id", how="left"))
lt_files = sheet_table.drop_duplicates(["file_id", "logical_table"]).groupby("file_id").n_files.max()
fa["files_sharing_its_structure"] = fa.file_id.map(lt_files).fillna(0).astype(int)
num_cols = ["n_copies", "total_rows", "max_sheet_rows", "total_cols", "blank_rows", "duplicate_rows",
            "mixed_type_cols", "empty_cols", "wide_layout_sheets", "lookup_table_candidates", "repeated_cells",
            "inconsistent_keys", "sheets_with_rows_elsewhere", "redundant_rows_est", "n_lookup_formula_cells",
            "n_conditional_agg_cells", "n_external_links", "n_formula_cells", "n_tables", "n_pivot_caches",
            "n_cross_sheet_formulas"]
for c in num_cols:
    fa[c] = pd.to_numeric(fa[c], errors="coerce").fillna(0).astype(np.int64)
for c in ["has_data_connections", "has_power_query", "truncated"]:
    fa[c] = fa[c].fillna(False).astype(bool)

fa["flag_exact_duplicate"] = fa.n_copies > 1
fa["flag_data_exists_elsewhere"] = fa.sheets_with_rows_elsewhere > 0
fa["flag_fragmented_table"] = fa.files_sharing_its_structure >= 2
fa["flag_over_25mb_excel_connector"] = (fa.ext == ".xlsx") & (fa.size_mb > EXCEL_CONNECTOR_MAX_MB)
fa["flag_over_2000_rows_powerapps"] = fa.max_sheet_rows > POWERAPPS_ROW_LIMIT
fa["flag_near_excel_row_limit"] = fa.max_sheet_rows >= 0.9 * EXCEL_MAX_ROWS
fa["flag_lookup_formulas"] = fa.n_lookup_formula_cells > 0
fa["flag_links_other_workbooks"] = fa.n_external_links > 0
fa["flag_repeated_attributes"] = fa.lookup_table_candidates > 0
fa["flag_mixed_type_columns"] = fa.mixed_type_cols > 0
fa["flag_wide_layout"] = fa.wide_layout_sheets > 0
fa["flag_version_style_name"] = _bool(fa.name_has_version_marker)
fa["flag_stale"] = fa.days_since_modified > STALE_AFTER_DAYS
fa["flag_unreadable"] = fa.status.eq("error")
flag_cols = [c for c in fa.columns if c.startswith("flag_")]
fa["n_flags"] = fa[flag_cols].sum(axis=1)
front = ["file_id", "rel_path", "file_name", "folder", "ext", "size_mb", "modified", "days_since_modified",
         "status", "error", "n_flags"] + flag_cols
fa = fa[front + [c for c in fa.columns if c not in front]].sort_values(["n_flags", "size_mb"], ascending=False)

# ---------- headline summary ----------
xl = fa[fa.ext == ".xlsx"]
ex_df = load_df(SP["excluded"])
gb = lambda mb: round(mb / 1e3, 2)
tot_sheets_data = int((sheets.n_rows > 0).sum())
lk_total_cells = int(lks.repeated_cells.sum()) if not lks.empty else 0
summary = [
    ("Scope", "Files audited (.csv/.xlsx, archive folders excluded)", len(fa), "", ""),
    ("Scope", "Total size (GB)", gb(fa.size_mb.sum()), "", ""),
    ("Scope", "Sheets containing data", tot_sheets_data, "", ""),
    ("Scope", "Rows of data across all sheets", int(sheets.n_rows.sum()), "", ""),
    ("Scope", "Archive folders skipped", len(ex_df) if ex_df is not None else 0,
     f"{int(ex_df.matching_files.sum()) if ex_df is not None and len(ex_df) else 0:,} files, "
     f"{gb(ex_df.size_mb.sum()) if ex_df is not None and len(ex_df) else 0} GB not audited", ""),
    ("Duplication", "Files that are byte-identical copies of another file", int(fa.flag_exact_duplicate.sum()),
     f"{len(exact_groups):,} groups; {gb(exact_groups.wasted_mb.sum())} GB in extra copies; "
     f"{int((exact_groups.distinct_names > 1).sum()):,} groups have different file names",
     "One table = one copy of the data; Dataverse duplicate detection rules"),
    ("Duplication", "Sheets holding identical data to another sheet (any name/format)",
     int(content_groups.n_sheets.sum() - len(content_groups)) if len(content_groups) else 0,
     f"{len(content_groups):,} groups; {int((content_groups.formats.str.contains(',')).sum()) if len(content_groups) else 0:,} "
     "span both .xlsx and .csv (exports/re-saves)", "Single source of truth"),
    ("Duplication", "Sheets whose rows mostly already exist in another file", len(red),
     f"~{int(red.redundant_rows_est.sum()):,} redundant rows ({len(overlaps):,} overlapping pairs)",
     "Roll-ups become queries/views instead of copied files"),
    ("Structure", "Logical tables split across 2+ files", len(multi),
     f"{int(multi.n_files.sum()):,} files; {int((multi.pattern == 'recurring drop (date-stamped files)').sum()):,} "
     "look like recurring date-stamped drops", "Append to one table via dataflows / Power Automate"),
    ("Structure", "Logical tables whose columns drift between files", int((multi.n_column_variants > 1).sum()),
     "renamed/added/removed columns, which force manual fixes before Power BI refreshes",
     "Fixed table schema with typed columns"),
    ("Structure", "Sheets in wide/pivoted layout (3+ date-like column headers)", int((sheets.period_header_cols >= 3).sum()),
     "must be unpivoted before Power BI can use them", "Long-format tables"),
    ("Relational", "Shared entity columns (same column in 3+ files)",
     int(shared.entity_candidate.sum()) if not shared.empty else 0,
     ("top: " + "; ".join(f"{r.example_header} ({r.n_files} files, {r.values_stored:,} values, ~{r.distinct_values_est:,} distinct)"
                          for r in shared[shared.entity_candidate].head(5).itertuples())) if not shared.empty else "",
     "One Dataverse table per entity, referenced by lookup columns"),
    ("Relational", "Sheets with repeated attributes (lookup-table candidates)", int((sheets.n_lookup_candidates > 0).sum()),
     f"{lk_total_cells:,} cells repeat a value that a related table would store once", "1:N relationships"),
    ("Relational", "Keys with conflicting attribute values (e.g. one ID, two spellings)", int(sheets.inconsistent_keys.sum()),
     "data-quality errors that a lookup makes impossible", "Lookups + required/validated columns"),
    ("Relational", "Lookup formula cells (VLOOKUP/XLOOKUP/INDEX-MATCH…)", int(xl.n_lookup_formula_cells.sum()),
     f"in {int(xl.flag_lookup_formulas.sum()):,} workbooks; plus {int(xl.n_conditional_agg_cells.sum()):,} SUMIFS/COUNTIFS-type cells",
     "Relationships defined once in the data model"),
    ("Relational", "Workbooks linked to other workbooks", int(xl.flag_links_other_workbooks.sum()),
     f"{int(xl.n_external_links.sum()):,} external links (break when files are renamed or moved)", "Relationships across tables"),
    ("Quality", "Columns mixing numbers/dates with text", int((cols.type_guess == "mixed").sum()),
     f"in {int(fa.flag_mixed_type_columns.sum()):,} files", "Typed columns, business rules"),
    ("Quality", "Duplicate rows inside sheets", int(sheets.duplicate_rows.sum()), "", "Alternate keys prevent duplicates"),
    ("Quality", "Files with version-style names (v2, final, copy, (1)…)", int(fa.flag_version_style_name.sum()),
     "manual version control", "Auditing / change history on every row"),
    ("Platform limits", f"Sheets over {POWERAPPS_ROW_LIMIT:,} rows", int((sheets.n_rows > POWERAPPS_ROW_LIMIT).sum()),
     f"Power Apps reads at most {POWERAPPS_ROW_LIMIT:,} rows from Excel (non-delegable)", "Delegable queries over large tables"),
    ("Platform limits", f"Workbooks over {EXCEL_CONNECTOR_MAX_MB} MB", int(fa.flag_over_25mb_excel_connector.sum()),
     "above the Excel Online (Business) connector's file size limit", "No file-size ceiling per table"),
    ("Platform limits", "Sheets at 90%+ of Excel's 1,048,576-row limit", int((sheets.n_rows >= 0.9 * EXCEL_MAX_ROWS).sum()),
     "", "Scales well past Excel's limit"),
    ("Platform limits", "Workbooks already using Power Query / data connections",
     int((xl.has_power_query | xl.has_data_connections).sum()), "existing transformation logic that could move to dataflows", "Dataflows"),
    ("Housekeeping", f"Files not modified in {STALE_AFTER_DAYS}+ days", int(fa.flag_stale.sum()),
     f"{gb(fa[fa.flag_stale].size_mb.sum())} GB", "Retention policies / bulk delete"),
    ("Housekeeping", "Files that couldn't be read", int(fa.flag_unreadable.sum()), "see `error` column in 01_file_audit.csv", ""),
]
summary_df = pd.DataFrame(summary, columns=["section", "metric", "value", "detail", "dataverse_capability"], dtype=object)

# ---------- manual-effort survey template, pre-filled with recurring drops ----------
survey_cols = ["process_name", "owner", "source (email / Power Apps / other)", "logical_table", "example_files",
               "runs_per_month", "minutes_per_run", "people_involved", "steps (copy-paste, reformat, rename columns, "
               "dedupe, lookups, fix errors)", "rework_minutes_per_month (fixing mistakes)", "powerbi_reports_fed", "notes"]
pre = frag[frag.n_files >= 2].head(25) if not frag.empty else frag
survey = pd.DataFrame([{survey_cols[3]: r.logical_table, survey_cols[4]: r.example_files} for r in pre.itertuples()],
                      columns=survey_cols)

# ---------- write ----------
OUT.mkdir(parents=True, exist_ok=True)
outputs = {
    "00_audit_summary.csv": summary_df,
    "01_file_audit.csv": fa,
    "02_exact_duplicate_files.csv": exact_groups.drop(columns=["sha256"]),
    "03_same_data_different_files.csv": content_groups.drop(columns=["content_key"]),
    "04_overlapping_datasets.csv": overlaps,
    "05_fragmented_tables.csv": frag.drop(columns=["schema_root"], errors="ignore"),
    "06_shared_entities.csv": shared,
    "07_lookup_table_candidates.csv": lks.merge(inv[["file_id", "rel_path"]], on="file_id", how="left")
                                         .sort_values("repeated_cells", ascending=False),
    "08_sheet_details.csv": sh.drop(columns=["row_sketch"]).merge(inv[["file_id", "rel_path"]], on="file_id", how="left"),
    "09_column_details.csv": cols.drop(columns=["col_sketch"]).merge(inv[["file_id", "rel_path"]], on="file_id", how="left"),
    "10_manual_effort_survey.csv": survey,
}
for name, df in outputs.items():
    save_df(df, OUT / name)
log(f"Wrote {len(outputs)} reports to {OUT}")
display(summary_df)

### Quick looks
The biggest offenders in each category. Open the CSVs in Excel for the full lists.

In [ ]:
show = lambda title, df, cols_=None, n=10: (print(f"\n=== {title} ==="), display((df[cols_] if cols_ else df).head(n)))
show("Most-flagged files", fa, ["rel_path", "size_mb", "n_flags"] + flag_cols[:6])
show("Exact duplicate groups", exact_groups, ["dup_group", "n_copies", "size_mb", "distinct_names", "files"])
show("Same data, different files", content_groups, ["content_group", "n_sheets", "rows", "formats", "locations"])
show("Overlapping datasets", overlaps, ["file_a", "sheet_name_a", "rows_a", "file_b", "sheet_name_b", "rows_b", "jaccard_est", "relation"])
if not frag.empty:
    show("Fragmented tables", frag, ["logical_table", "pattern", "n_files", "n_column_variants", "total_rows", "drifting_columns"])
if not shared.empty:
    show("Shared entities", shared[shared.entity_candidate], ["example_header", "n_files", "values_stored", "distinct_values_est", "repetition_factor"])
show("Lookup-table candidates", outputs["07_lookup_table_candidates.csv"], ["rel_path", "sheet_name", "key_column", "dependent_columns", "repeated_cells", "inconsistent_keys"])